In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
data_path = "dataset/"
files = {}
for f in os.listdir(data_path):
    name = f.replace(".csv", "")
    files[name] = pd.read_csv(data_path + f)
    print(f"{f}: {files[name].shape}")

In [ ]:
sales       = files['sales'].copy()
orders      = files['orders'].copy()
order_items = files['order_items'].copy()
products    = files['products'].copy()
customers   = files['customers'].copy()
geography   = files['geography'].copy()
returns     = files['returns'].copy()
promotions  = files['promotions'].copy()
web_traffic = files['web_traffic'].copy()
inventory   = files['inventory'].copy()
 
sales['Date']      = pd.to_datetime(sales['Date'])
orders['order_date'] = pd.to_datetime(orders['order_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date']   = pd.to_datetime(promotions['end_date'])
web_traffic['date']      = pd.to_datetime(web_traffic['date'])
 
sales['year']  = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
sales['gross_margin'] = (sales['Revenue'] - sales['COGS']) / sales['Revenue']

In [ ]:
files['sales']['gross_margin'] = (files['sales']['Revenue'] - files['sales']['COGS']) / files['sales']['Revenue']
files['sales']['Date'] = pd.to_datetime(files['sales']['Date'])

In [ ]:
files['promotions']['start_date'] = pd.to_datetime(files['promotions']['start_date'])
files['promotions']['end_date'] = pd.to_datetime(files['promotions']['end_date'])
files['promotions']['duration'] = (files['promotions']['end_date'] - files['promotions']['start_date']).dt.total_seconds() * 1000

In [ ]:
daily = files['sales'].copy()

In [ ]:
def get_discount(current_date):
    # Find rows where the date is within the promotion range
    mask = (files['promotions']['start_date'] <= current_date) & (current_date <= files['promotions']['end_date'])
    valid_promos = files['promotions'].loc[mask]

    if not valid_promos.empty:
        # Return the max discount if multiple promotions overlap
        return valid_promos['discount_value'].max()
    return 0

In [ ]:
daily['promotions_discount_value'] = daily['Date'].apply(get_discount)

In [ ]:
files['web_traffic']['date'] = pd.to_datetime(files['web_traffic']['date'])
daily = daily.merge(files['web_traffic'][['date', 'sessions', 'traffic_source']], left_on='Date', right_on='date', how='left')

In [ ]:
daily['day_of_week'] = daily['Date'].dt.day_name()
daily['month'] = daily['Date'].dt.month_name()

In [ ]:
merged_order = pd.merge(files['orders'], files['order_items'], left_on='order_id', right_on='order_id', how='left')
merged_order['order_date'] = pd.to_datetime(merged_order['order_date'])
merged_order = merged_order.merge(files['geography'][['zip', 'city']], left_on='zip', right_on='zip', how='left')
merged_order = merged_order.merge(files['customers'][['customer_id', 'gender', 'age_group']], left_on='customer_id', right_on='customer_id', how='left')
merged_order = merged_order.merge(files['products'][['product_id', 'category', 'segment']], left_on='product_id', right_on='product_id', how='left')
merged_order = merged_order.merge(files['reviews'][['order_id', 'rating']], left_on='order_id', right_on='order_id', how='left')
merged_order = merged_order.merge(files['returns'][['order_id', 'return_reason']], left_on='order_id', right_on='order_id', how='left')
merged_order['discount_percentage'] = merged_order['discount_amount']/ (merged_order['unit_price'] + merged_order['discount_amount'])
merged_order['order_date'] = pd.to_datetime(merged_order['order_date'])

merged_order.drop(columns=['zip', 'order_status', 'payment_method', 'order_source'], inplace=True)

In [ ]:
daily['sessions'] = daily['sessions'].fillna(daily['sessions'].mean())
daily['traffic_source'] = daily['traffic_source'].fillna('Unknown')

In [ ]:
temp = daily.copy()
temp.groupby('traffic_source')['sessions'].sum().reset_index().sort_values('sessions', ascending=False)

In [ ]:
px.histogram(temp, x='traffic_source', y='sessions', color='traffic_source')

In [ ]:
daily['gross_margin'] = 100 * daily['gross_margin']
daily.drop(columns=['date'], inplace=True)
daily

In [ ]:
merged_order['revenue'] = merged_order['unit_price'] * merged_order['quantity']
merged_order['rating'] = merged_order['rating'].fillna(0)
merged_order['return_reason'] = merged_order['return_reason'].fillna('Not Returned')

In [ ]:
merged_order['day_of_week'] = merged_order['order_date'].dt.day_name()
merged_order['month'] = merged_order['order_date'].dt.month_name()

In [ ]:
temp1 = merged_order['day_of_week'].value_counts()
temp1

# Graphing

## Gross Margin with Promotions

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Scatter(x=daily['Date'], y=daily['gross_margin'], name='Gross Margin'), secondary_y=False)
fig.add_trace(go.Bar(x=files['promotions']['start_date'], y=files['promotions']['discount_value'], width=files['promotions']['duration'], offset=0, opacity=0.4, yaxis='y3', marker_color='green', name='Promotions'), secondary_y=True)

fig.update_xaxes(
    tickformat='%b %Y',
    dtick='M1',
    tickfont=dict(size=6)
)
fig.update_layout(yaxis_ticksuffix='%', yaxis2_ticksuffix='%', title='Gross Margin and Promotions Over Time')
fig.show()

## Chart 1 — Gross Margin and Promotions Over Time
 
**Descriptive — What happened?**
Gross margin generally hovers between 15–25% across 2012–2022, but experiences dramatic periodic crashes down to -40% or lower. These crashes are sharp, short-lived, and occur roughly every 1–2 years.
 
**Diagnostic — Why did it happen?**
Every major margin crash coincides precisely with a tall promotion bar (50% discount value). The crashes are not gradual — they are caused by high fixed-value or high-percentage promotions that push unit economics below cost. Normal promotions (10–20% discount) barely move the margin line; only the largest campaigns cause these collapses.
 
**Predictive — What is likely to happen?**
The pattern is highly regular — a major margin-destroying promotion appears roughly every 18 months. If this cadence continues, the business should anticipate another crash in early-to-mid 2023. The end-of-2022 trend already shows margin declining toward 0%.
 
**Prescriptive — What should we do?**
The 50%+ discount promotions are value-destroying. The business should set a hard cap on discount depth (e.g. maximum 30%), or restrict deep discounts only to products with above-average margin. Running a promotion does not justify selling below cost — these campaigns are effectively paying customers to buy.

## Number of order wrt some interesting things

In [ ]:

# 1. Ensure order_date is a datetime object
# 2. Extract Year and Month (as Name)
merged_order['year'] = merged_order['order_date'].dt.year


# 3. Group by Year and Month to get the count for every individual month in your data
# This creates a table like: [2023, January, 50], [2024, January, 60]
yearly_counts = merged_order.groupby(['year', 'month']).size().reset_index(name='order_count')

# 4. Plot the average of those counts
fig = px.histogram(
    yearly_counts, 
    x='month', 
    y='order_count', 
    histfunc='avg', 
    color='month',
    title='Average Monthly Orders (Year-over-Year)'
)

# 5. Handle sorting (Optional: ensures Jan -> Dec order)
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

fig.update_xaxes(categoryorder='array', categoryarray=month_order)
fig.show()

## Chart 2 — Average Monthly Orders (Year-over-Year)
 
**Descriptive — What happened?**
April and May are the clear peak months, averaging ~8,200 and ~8,134 orders per year respectively. June (~7,800) and August (~6,600) are the next strongest. The year follows a bimodal shape — a major peak in April–June, a partial recovery in July–August, then a steady decline through September–November (bottoming at ~3,800), with a modest December rebound to ~5,800.
 
**Diagnostic — Why did it happen?**
The April–June peak aligns with the spring/summer fashion season, end-of-school-year spending, and the lead-up to Vietnam's summer holidays. The December partial recovery (~5,800) likely reflects year-end gifting and the 12.12 sale event. The October–November trough is surprising — this is globally a peak period (11.11, Black Friday) but appears weak here, suggesting the business has not successfully capitalised on those campaigns or its customer base is not responsive to them.
 
**Predictive — What is likely to happen?**
The April–June peak is the most reliable demand signal in the data. Inventory and logistics capacity must be scaled up by late February/early March to be ready. January (~3,300) and February (~4,000) represent the floor — these are structurally weak months that promotions alone are unlikely to fix.
 
**Prescriptive — What should we do?**
Aggressively invest in 11.11 and Black Friday campaigns to capture the October–November gap — these months are currently underperforming global e-commerce norms and represent recoverable revenue. For April–June, focus on margin protection rather than promotions since demand is already at its natural peak. Use January–February's low-demand period for inventory clearance of prior-season stock at controlled discount depths.
 

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Order Distribution by Category and Segment",
    "Revenue Distribution by Category and Segment"
])

# First heatmap (order distribution)
fig1 = px.density_heatmap(
    merged_order, x='category', y='segment',
    color_continuous_scale='Greens'
).update_traces(histnorm='percent', texttemplate='%{z:.2f}%')

# Second heatmap (revenue distribution)
fig2 = px.density_heatmap(
    merged_order, x='category', y='segment',
    z='revenue', histfunc='sum',
    color_continuous_scale='Greens'
).update_traces(histnorm='percent', texttemplate='%{z:.2f}%')

# Add traces to subplots
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(height=500, width=1000, coloraxis=dict(colorscale='Greens'), title="Order and Revenue Distribution by Category and Segment")

fig.show()

## Chart 3 — Order and Revenue Distribution by Category × Segment
 
**Descriptive — What happened?**
The distribution is highly concentrated. Outdoor × Activewear dominates order count at 33%. On the revenue side, Streetwear × Everyday (32.54%) and Streetwear × Balanced (31.02%) together make up nearly 64% of all revenue — an extreme concentration in a single category.
 
**Diagnostic — Why did it happen?**
Comparing both heatmaps reveals a key insight: Outdoor × Activewear has 33% of orders but only 12.74% of revenue, meaning its average order value is significantly lower than its volume suggests. Meanwhile Streetwear × Everyday and Streetwear × Balanced have ~39% of orders but ~64% of revenue — customers in these segments spend nearly twice as much per order. GenZ × Performance contributes a balanced 13.34% of orders and 14.47% of revenue, suggesting consistent mid-range pricing.
 
**Predictive — What is likely to happen?**
The business is heavily exposed to the Streetwear category. Any trend shift away from streetwear, or a supply disruption in that category, would devastate ~63% of revenue with no other category large enough to compensate.
 
**Prescriptive — What should we do?**
Reduce concentration risk by investing in growing the Outdoor × Activewear segment's average order value through product bundling and accessory upsells. The Premium and All-weather segments are near-zero in both orders and revenue — either discontinue them or run a targeted acquisition campaign to test viability. Streetwear should be protected as the core revenue engine with guaranteed inventory depth and first-priority restocking.

In [ ]:
px.density_heatmap(
    merged_order[merged_order['return_reason'] != 'Not Returned'], 
    x='category', 
    y='segment', 
    text_auto=True,                
    color_continuous_scale='Reds', 
    title='Total Return Counts by Category and Segment'
)

## Chart 4 — Total Return Counts by Category and Segment
 
**Descriptive — What happened?**
Returns are dominated by two cells: Outdoor × Activewear with 20,049 returns (by far the highest), and Streetwear × Everyday with 10,438 returns. Streetwear × Balanced (5,749) and GenZ × Performance (5,622) are the next largest. Casual and Premium segments have minimal returns.
 
**Diagnostic — Why did it happen?**
Cross-referencing with Chart 3, Outdoor × Activewear has 33% of orders but 20k returns — a disproportionately high return rate relative to its revenue contribution (only 12.74%). This likely reflects sizing complexity in activewear (tight-fit products carry higher sizing uncertainty). Streetwear's returns are more proportional to its volume, but still significant in absolute terms.
 
**Predictive — What is likely to happen?**
Without intervention, Outdoor × Activewear returns will continue eroding net revenue. Each return carries reverse logistics costs, refund processing, and restocking overhead — at scale, this turns an already low-revenue-per-order category into a potential net loss segment.
 
**Prescriptive — What should we do?**
Prioritise adding detailed size charts, fit guides, and customer-submitted photos specifically for the Activewear segment. Consider implementing an exchange-first returns policy (rather than immediate refund) to reduce net cash outflow. For Streetwear × Everyday, cross-reference with the return reasons chart to determine whether the fix is operational (sizing) or marketing (expectation management).

In [ ]:
px.histogram(merged_order.sort_values(by=['age_group']), x='age_group', y='revenue', title='Age - Revenue Distribution', histnorm='percent', range_y=[0, 40]).update_traces(texttemplate='%{y:.1f}%', textposition='outside').update_traces(marker_color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A'])

## Chart 5 — Age - Revenue Distribution
 
**Descriptive — What happened?**
The 25–34 age group generates the largest share of revenue at 29.5%, followed by 35–44 at 26.3%, 45–54 at 19.3%, 18–24 at 13.7%, and 55+ at 11.2%. The two middle brackets (25–44) together account for 55.8% of all revenue.
 
**Diagnostic — Why did it happen?**
The 18–24 group underperforms relative to its population size in Vietnam — this age group is large and digitally native, yet contributes the least of the younger segments. This likely reflects lower disposable income and higher price sensitivity. The 55+ group's 11.2% is notable and should not be ignored — it suggests older demographics are actively engaging with the platform despite lower digital adoption expectations.
 
**Predictive — What is likely to happen?**
Vietnam's 25–34 cohort will age into the 35–44 bracket over the next decade, likely maintaining or increasing their spending power. The current 18–24 cohort is the future 25–34 group — acquiring them now at a lower customer acquisition cost could yield significant long-term returns.
 
**Prescriptive — What should we do?**
For 25–34 and 35–44, focus on **retention** — loyalty points, personalised recommendations, and early access to new collections to protect the core revenue base. For 18–24, offer entry-level price points and installment payment options (BNPL) to reduce the conversion barrier. Do not neglect 55+: this segment likely has high average order value and low return rates — consider a curated "classic" product line targeting their preferences.

In [ ]:
temp = merged_order['order_date'].value_counts().reset_index(name='count')
temp.sort_values('order_date', inplace=True)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=temp['order_date'], y=temp['count'], name='Orders by Date'), secondary_y=False)
fig.add_trace(go.Bar(x=files['promotions']['start_date'], y=files['promotions']['discount_value'], width=files['promotions']['duration'], offset=0, opacity=0.4, yaxis='y3', marker_color='green', name='Promotions'), secondary_y=True)

fig.update_xaxes(
    tickformat='%b %Y',
    dtick='M1',
    tickfont=dict(size=6)
)
fig.update_layout(yaxis2_ticksuffix='%', title='Orders and Promotions Over Time')
fig.show()

## Chart 6 — Orders and Promotions Over Time
 
**Descriptive — What happened?**
Daily order counts show raw numbers ranging from 0–900 orders per day. The business peaked around 2013–2017, with regular daily spikes of 600–900 orders. From 2018 onward, both the baseline and spike heights decline noticeably — by 2021–2022, most days sit below 200 orders with occasional spikes to 400. The largest promotion bars (50% discount, the tall green spikes) appear roughly every 18 months.
 
**Diagnostic — Why did it happen?**
The 50% promotion spikes do not consistently produce order spikes above the surrounding noise — in several cases (e.g. mid-2019, mid-2021), a large promotion bar coincides with a period of already-low orders, suggesting promotions were launched reactively to try to stimulate a declining trend rather than amplifying an existing peak. The genuine order spikes (800–900 orders/day) in 2013–2017 appear to be organic demand, not promotion-driven. This reinforces the finding from Chart 8 — promotions are not the engine of order volume.
 
**Predictive — What is likely to happen?**
The structural decline in daily order volume from ~400 baseline in 2015 to ~100 baseline in 2022 is an alarming trend. Without a fundamental change in customer acquisition or product-market fit, 2023 order volumes will likely continue at the post-2019 lower level.
 
**Prescriptive — What should we do?**
Stop using promotions as a reactive tool to fight declining demand — the data shows it does not work. Instead, investigate what drove the organic 600–900 order days in 2013–2017: was it a specific product launch, a marketing campaign, or an external trend? Replicating those conditions is more valuable than discounting. In parallel, implement a customer reactivation program targeting the large cohort of customers who were active in 2013–2017 but have since lapsed.

In [ ]:
monthly = sales.groupby(['year', 'month'])['Revenue'].sum().reset_index()
pivot   = monthly.pivot(index='year', columns='month', values='Revenue')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig1 = px.imshow(
    pivot,
    color_continuous_scale='Greens',
    aspect='auto',
    title='<b>Monthly Revenue Heatmap (2012–2022)</b><br><sup>Darker green = higher revenue; reveals seasonality & year-over-year growth</sup>',
    labels=dict(x='Month', y='Year', color='Revenue (VND)')
)
fig1.update_layout(width=900, height=450, coloraxis_colorbar_title='Revenue')
fig1.show()

## Chart 7 — Monthly Revenue Heatmap (2012–2022)
 
**Descriptive — What happened?**
Revenue was highest during 2014–2018, particularly in the April–June window (deep green, 200–270M VND). From 2019 onward, the entire heatmap shifts to red — revenue declined significantly across all months and has not recovered. The 2020 row is uniformly dark red, and 2022 shows similarly weak performance at 50–150M VND across most months.
 
**Diagnostic — Why did it happen?**
The 2020 revenue collapse is partially explained by COVID-19 disrupting logistics, consumer confidence, and supply chains. However, the decline began in 2019 — pre-COVID — suggesting a structural issue preceded the pandemic. Possible causes include market saturation, increased competition from platforms like Shopee and Lazada, or a product mix that stopped resonating with customers.
 
**Predictive — What is likely to happen?**
The heatmap shows no recovery trend through 2022. Revenues remain in the 50–150M VND range compared to 200–270M+ in peak years. Without a strategic intervention, 2023 forecasts should be anchored to the post-2019 lower revenue regime rather than assuming a reversion to peak.
 
**Prescriptive — What should we do?**
Conduct a root-cause analysis of the 2019 inflection point — compare 2018 vs 2019 new customer acquisition, retention rates, and category mix. If customer attrition is the driver, a win-back campaign targeting customers whose last order was pre-2019 could recover meaningful revenue at low cost. If the issue is competitive displacement, a pricing and assortment review against major competitors is needed before any marketing investment.

In [ ]:
def is_promo_day(date):
    return ((promotions['start_date'] <= date) & (date <= promotions['end_date'])).any()

sales['promo_active'] = sales['Date'].apply(is_promo_day)

promo_compare = sales.groupby('promo_active').agg(
    avg_revenue=('Revenue', 'mean'),
    avg_margin=('gross_margin', 'mean'),
    days=('Revenue', 'count')
).reset_index()
promo_compare['label'] = promo_compare['promo_active'].map({True: 'Promo Day', False: 'Non-Promo Day'})

fig5 = make_subplots(rows=1, cols=2,
    subplot_titles=('Average Daily Revenue', 'Average Gross Margin %'))

colors = ['#E53935', '#43A047']
fig5.add_trace(go.Bar(
    x=promo_compare['label'], y=promo_compare['avg_revenue'],
    marker_color=colors, showlegend=False
), row=1, col=1)

fig5.add_trace(go.Bar(
    x=promo_compare['label'], y=promo_compare['avg_margin'],
    marker_color=colors, showlegend=False
), row=1, col=2)

fig5.update_yaxes(tickformat='.1%', row=1, col=2)
fig5.update_layout(
    title='<b>Promotion Days vs. Non-Promotion Days</b><br>'
          '<sup>Are revenue gains worth the margin sacrifice?</sup>',
    width=800, height=450
)
fig5.show()

## Chart 8 — Promotion Days vs. Non-Promotion Days
 
**Descriptive — What happened?**
Promotion days generate ~4M VND average daily revenue vs ~4M on non-promotion days — effectively identical. However, gross margin on promotion days collapses to ~3.5% compared to ~20% on non-promotion days — a reduction of over 80%.
 
**Diagnostic — Why did it happen?**
This is the single most important finding in the entire EDA. Promotions are **revenue-neutral but margin-destructive**. The business is running campaigns that cost it nearly all gross profit without generating any meaningful additional sales volume. Customers who would have bought anyway are simply buying at a lower price.
 
**Predictive — What is likely to happen?**
At 3.5% gross margin during promotions, after accounting for operating costs (logistics, customer service, returns processing), promotion days are almost certainly operating at a **net loss**. If promotions run on approximately 20–25% of days annually, they are dragging the overall business margin down by several percentage points every year.
 
**Prescriptive — What should we do?**
This chart alone justifies an immediate review of the promotions strategy. Specifically: (1) **Halt all promotions above 30% discount depth** pending further analysis. (2) **Run a controlled hold-out test** — eliminate promotions for one quarter in a subset of customer segments and measure whether revenue actually drops. Given that promo-day and non-promo-day revenues are equal, the hypothesis that promotions are not driving incremental demand is strongly supported. (3) **Redirect promotion budget** toward loyalty rewards and personalised discounts for high-value customers rather than blanket sitewide discounts that cannabilise full-price sales.

In [ ]:
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
inventory['year']  = inventory['snapshot_date'].dt.year
inventory['month_num'] = inventory['snapshot_date'].dt.month

# Monthly stockout rate
stockout_monthly = inventory.groupby(['year','month_num']).agg(
    total_products=('product_id','count'),
    stockout_products=('stockout_flag','sum'),
    avg_fill_rate=('fill_rate','mean')
).reset_index()
stockout_monthly['stockout_rate'] = stockout_monthly['stockout_products'] / stockout_monthly['total_products']
stockout_monthly['date'] = pd.to_datetime(stockout_monthly[['year','month_num']].rename(columns={'month_num':'month'}).assign(day=1))

monthly_rev = sales.groupby(sales['Date'].dt.to_period('M'))['Revenue'].sum().reset_index()
monthly_rev['date'] = monthly_rev['Date'].dt.to_timestamp()

merged_stock = stockout_monthly.merge(monthly_rev[['date','Revenue']], on='date', how='inner')

fig7 = make_subplots(specs=[[{"secondary_y": True}]])
fig7.add_trace(go.Scatter(
    x=merged_stock['date'], y=merged_stock['stockout_rate'],
    name='Stockout Rate', mode='lines+markers',
    line=dict(color='crimson', width=2)
), secondary_y=False)
fig7.add_trace(go.Bar(
    x=merged_stock['date'], y=merged_stock['Revenue'],
    name='Monthly Revenue', marker_color='steelblue', opacity=0.5
), secondary_y=True)

fig7.update_layout(
    title='<b>Stockout Rate vs. Monthly Revenue</b><br>'
          '<sup>Do stockout spikes coincide with revenue dips? Quantifies inventory risk</sup>',
    width=1100, height=500, hovermode='x unified'
)
fig7.update_yaxes(title_text='Stockout Rate', tickformat='.0%', secondary_y=False)
fig7.update_yaxes(title_text='Revenue (VND)', secondary_y=True)
fig7.show()

## Chart 9 — Stockout Rate vs. Monthly Revenue
 
**Descriptive — What happened?**
The stockout rate is persistently high, fluctuating between 58–72% throughout the entire 2012–2022 period — meaning at any given month, roughly 2 in 3 products experienced at least one day out of stock. Monthly revenue peaked around 2016–2018 at 250–270M VND and has declined since, sitting around 50–150M VND by 2022.
 
**Diagnostic — Why did it happen?**
Critically, stockout rate and revenue move **independently** — there is no visible inverse correlation. Revenue declined sharply from 2019 onward while stockout rate remained in the same 60–72% band it always occupied. This tells us the revenue collapse was **not caused by stockouts** — the inventory problem existed even during high-revenue years and customers still bought. The real driver of revenue decline must lie elsewhere.
 
**Predictive — What is likely to happen?**
With stockout rate showing no improvement over 10 years and revenue continuing to decline, the business is entering 2023 with both chronic inventory inefficiency and weakening demand simultaneously — a compounding risk that will worsen without intervention.
 
**Prescriptive — What should we do?**
Since stockouts did not cause the revenue decline, fixing stockouts alone will not recover revenue. However, a sustained 65%+ stockout rate is still operationally unacceptable — customers regularly encountering out-of-stock pages damages brand trust over time. Set a target to bring the stockout rate below 40% within 12 months by improving demand forecasting, particularly for the Outdoor × Activewear segment. Address the revenue decline separately through the customer cohort and competitive analysis recommended in Chart 7.

In [ ]:
orders_geo = orders.merge(geography[['zip','region','city']], on='zip', how='left')
orders_items_geo = order_items.merge(orders_geo[['order_id','region','city']], on='order_id', how='left')
orders_items_geo['line_revenue'] = orders_items_geo['quantity'] * orders_items_geo['unit_price']

region_rev = orders_items_geo.groupby('region')['line_revenue'].sum().reset_index().sort_values('line_revenue', ascending=False)
city_rev   = orders_items_geo.groupby(['region','city'])['line_revenue'].sum().reset_index().sort_values('line_revenue', ascending=False).head(15)

fig4 = make_subplots(rows=1, cols=2,
    subplot_titles=('Revenue by Region', 'Top 15 Cities by Revenue'))

fig4.add_trace(go.Bar(
    x=region_rev['region'], y=region_rev['line_revenue'],
    marker_color=['#2196F3','#FF9800','#4CAF50'],
    showlegend=False
), row=1, col=1)

fig4.add_trace(go.Bar(
    x=city_rev['line_revenue'], y=city_rev['city'],
    orientation='h',
    marker_color='steelblue', showlegend=False
), row=1, col=2)

fig4.update_layout(
    title='<b>Revenue by Geography</b><br>'
          '<sup>Identifies high-value markets for targeted inventory & marketing spend</sup>',
    width=1100, height=500
)
fig4.show()


## Chart 10 — Revenue by Geography
 
**Descriptive — What happened?**
The East region dominates with ~7.5B VND total revenue, nearly 1.5x Central (~5B) and almost 2x West (~3.9B). In the top 15 cities, the ranking is surprisingly tight — Son Tay leads at ~590M VND with all 15 cities clustered in a narrow 460–590M VND band. Notably, Hanoi appears mid-list alongside smaller cities like Kon Tum, Lao Cai, and Uong Bi.
 
**Diagnostic — Why did it happen?**
The presence of small provincial cities (Kon Tum, Lao Cai, Uong Bi) generating revenue comparable to Hanoi is unexpected. This may reflect a small number of very high-value customers in those cities driving disproportionate revenue, or noise in the zip-to-city mapping. The East region's dominance reflects northern Vietnam's economic concentration and population density. The tight clustering of top-15 cities means the business has no dangerous over-reliance on a single city.
 
**Predictive — What is likely to happen?**
As e-commerce penetration grows in provincial Vietnam, the contribution from cities like Bac Ninh, Bac Giang, and Viet Tri (industrial zones with rising incomes) is likely to increase. The West region's underperformance relative to its population base represents an untapped growth opportunity.
 
**Prescriptive — What should we do?**
Prioritise logistics and same-day delivery capabilities in the East region, which generates nearly double the West's revenue. Investigate the high-revenue provincial cities — if they are driven by a small number of bulk buyers, assign dedicated account managers to protect that revenue. Launch a targeted acquisition campaign in the West region to close the gap with East and Central, potentially starting with Streetwear (the highest-revenue category) as the entry product.

In [ ]:
sales['day_of_week'] = sales['Date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow_stats = sales.groupby('day_of_week').agg(
    avg_revenue=('Revenue','mean'),
    median_revenue=('Revenue','median')
).reindex(dow_order).reset_index()

fig9 = go.Figure()
fig9.add_trace(go.Bar(
    x=dow_stats['day_of_week'], y=dow_stats['avg_revenue'],
    name='Avg Revenue', marker_color='steelblue'
))
fig9.add_trace(go.Scatter(
    x=dow_stats['day_of_week'], y=dow_stats['median_revenue'],
    name='Median Revenue', mode='lines+markers',
    line=dict(color='orange', width=2)
))
fig9.update_layout(
    title='<b>Average Revenue by Day of Week</b><br>'
          '<sup>Gap between mean and median reveals high-value outlier days; '
          'use for flash sale scheduling</sup>',
    xaxis_title='Day of Week',
    yaxis_title='Revenue (VND)',
    width=800, height=450
)
fig9.show()

## Chart 11 — Average Revenue by Day of Week
 
**Descriptive — What happened?**
Wednesday and Thursday are the strongest days, averaging ~4.45–4.52M VND respectively. Monday and Tuesday are slightly lower at ~4.2–4.4M. The weekend is noticeably weaker — Friday dips to ~4.0M, Saturday to ~3.9M, and Sunday recovers slightly to ~4.05M. The median line (orange) consistently sits below the average on every day, indicating right-skewed distributions — a few very high-revenue days pull each daily average upward.
 
**Diagnostic — Why did it happen?**
The mid-week peak suggests the core customer is a working professional who browses and purchases during the workweek, likely during lunch breaks or after-work hours. The weekend dip is counter-intuitive for fashion e-commerce and may indicate that the business's primary acquisition channels (email, search ads) perform better on weekdays when office workers are online and checking inboxes. The mean-median gap is largest on Wednesday–Thursday, likely driven by mid-week promotions creating occasional revenue spikes.
 
**Predictive — What is likely to happen?**
The weekday-heavy pattern will likely persist as long as the customer base is predominantly working-age professionals. Weekend revenue will remain structurally lower unless deliberate steps are taken to engage customers on their days off.
 
**Prescriptive — What should we do?**
Schedule email marketing campaigns and flash sale launches for **Tuesday–Thursday** to capitalise on peak intent days. Avoid starting new promotions on Fridays or weekends when baseline engagement is already lower — this wastes promotional budget. To lift weekend revenue, invest in weekend-specific content such as styling guides, outfit lookbooks, and social media campaigns targeting leisure browsing behaviour on Saturday mornings.
 

In [ ]:
returns_full = returns.merge(products[['product_id','category']], on='product_id', how='left')
return_reasons = returns_full.groupby(['category','return_reason'])['return_quantity'].sum().reset_index()

fig10 = px.bar(
    return_reasons,
    x='category', y='return_quantity', color='return_reason',
    barmode='stack',
    title='<b>Return Reasons by Product Category</b><br>'
          '<sup>wrong_size → improve size guides; defective → supplier quality issue; '
          'changed_mind → pricing/expectation mismatch</sup>',
    labels=dict(return_quantity='Returned Units', category='Category')
)
fig10.update_layout(width=1000, height=500)
fig10.show()

## Chart 12 — Return Reasons by Product Category
 
**Descriptive — What happened?**
Streetwear has the highest total returns at ~60k units, followed by Outdoor at ~40k. Casual (~3k) and GenZ (~6k) have minimal returns. In both Streetwear and Outdoor, `wrong_size` (orange) is the single largest return reason, making up the top 35–40% of each bar. `defective` (red) is the second largest in both categories, followed by `not_as_described` (purple), `late_delivery` (green), and `changed_mind` (blue) at the base.
 
**Diagnostic — Why did it happen?**
`wrong_size` being the #1 reason across both high-volume categories is a clear operational signal — the business's size guides are inadequate for these product types. Streetwear and Outdoor garments (hoodies, joggers, technical outerwear) have more complex fit than basic casual wear, and customers are guessing wrong at scale. `defective` being the second largest reason — particularly prominent in Streetwear at ~12k units — points to a **supplier quality control problem**, not a customer behaviour issue. The low return rates in Casual and GenZ categories set a useful internal benchmark.
 
**Predictive — What is likely to happen?**
Without intervention, return volumes will grow proportionally with sales. Given that Streetwear is the highest-revenue category (64% of revenue), even a modest improvement in its return rate would have a significant impact on net revenue. Each percentage point reduction in Streetwear returns saves thousands of units in reverse logistics costs annually.
 
**Prescriptive — What should we do?**
Three targeted actions, each addressing a specific return reason:
1. **wrong_size** — Implement a size recommendation tool (e.g. height/weight-based fit quiz) specifically for Streetwear and Outdoor. Even a 20% reduction in size-related returns would recover thousands of units per year.
2. **defective** — Introduce incoming quality inspection for the top Streetwear suppliers and add a defect rate clause with financial penalties to supplier contracts.
3. **not_as_described** — Audit product photography and copy for Outdoor items, focusing on material texture, fit accuracy, and realistic colour representation.
Use Casual and GenZ's low return rates as an internal benchmark — study what those categories do differently (simpler sizing? better product descriptions?) and replicate it in Streetwear and Outdoor.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sales = files['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

# ── Compute rolling variance (30-day window) ─────────────────
window = 30
sales['rev_variance']  = sales['Revenue'].rolling(window).var()
sales['cogs_variance'] = sales['COGS'].rolling(window).var()
sales['rev_std']       = sales['Revenue'].rolling(window).std()
sales['cogs_std']      = sales['COGS'].rolling(window).std()

# ── Rolling mean (for the confidence band) ───────────────────
sales['rev_mean']  = sales['Revenue'].rolling(window).mean()
sales['cogs_mean'] = sales['COGS'].rolling(window).mean()

# ══════════════════════════════════════════════════════════════
# CHART 1 — Revenue & COGS with ±1 STD confidence band
# Shows where variance is high (wide band) vs stable (narrow band)
# ══════════════════════════════════════════════════════════════

fig1 = go.Figure()

# Revenue band
fig1.add_trace(go.Scatter(
    x=pd.concat([sales['Date'], sales['Date'][::-1]]),
    y=pd.concat([
        sales['rev_mean'] + sales['rev_std'],
        (sales['rev_mean'] - sales['rev_std'])[::-1]
    ]),
    fill='toself',
    fillcolor='rgba(33, 150, 243, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Revenue ±1 STD',
    showlegend=True
))

# COGS band
fig1.add_trace(go.Scatter(
    x=pd.concat([sales['Date'], sales['Date'][::-1]]),
    y=pd.concat([
        sales['cogs_mean'] + sales['cogs_std'],
        (sales['cogs_mean'] - sales['cogs_std'])[::-1]
    ]),
    fill='toself',
    fillcolor='rgba(244, 67, 54, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='COGS ±1 STD',
    showlegend=True
))

# Revenue line
fig1.add_trace(go.Scatter(
    x=sales['Date'], y=sales['Revenue'],
    mode='lines',
    name='Revenue',
    line=dict(color='#2196F3', width=1.2)
))

# COGS line
fig1.add_trace(go.Scatter(
    x=sales['Date'], y=sales['COGS'],
    mode='lines',
    name='COGS',
    line=dict(color='#F44336', width=1.2)
))

fig1.update_layout(
    title='<b>Revenue & COGS with Rolling Variance Band (30-day window)</b><br>'
          '<sup>Wide band = high variance period; narrow band = stable period</sup>',
    xaxis_title='Date',
    yaxis_title='Value (VND)',
    hovermode='x unified',
    width=1100, height=500,
    legend=dict(orientation='h', y=1.08)
)
fig1.show()


# ══════════════════════════════════════════════════════════════
# CHART 2 — Rolling Variance over Time (separate subplots)
# Shows directly how variance evolves across years
# ══════════════════════════════════════════════════════════════

fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'Revenue — 30-Day Rolling Variance',
        'COGS — 30-Day Rolling Variance'
    ),
    vertical_spacing=0.1
)

fig2.add_trace(go.Scatter(
    x=sales['Date'], y=sales['rev_variance'],
    mode='lines',
    name='Revenue Variance',
    line=dict(color='#2196F3', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(33, 150, 243, 0.1)'
), row=1, col=1)

fig2.add_trace(go.Scatter(
    x=sales['Date'], y=sales['cogs_variance'],
    mode='lines',
    name='COGS Variance',
    line=dict(color='#F44336', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(244, 67, 54, 0.1)'
), row=2, col=1)

fig2.update_layout(
    title='<b>Rolling Variance Over Time (30-day window)</b><br>'
          '<sup>Spikes indicate high-volatility periods — check against promotions & seasonal events</sup>',
    width=1100, height=600,
    hovermode='x unified'
)
fig2.update_yaxes(title_text='Variance', row=1, col=1)
fig2.update_yaxes(title_text='Variance', row=2, col=1)
fig2.update_xaxes(title_text='Date', row=2, col=1)
fig2.show()


# ══════════════════════════════════════════════════════════════
# CHART 3 — Coefficient of Variation (CV = std/mean)
# Normalised volatility — lets you compare Revenue vs COGS fairly
# even though they're on different scales
# ══════════════════════════════════════════════════════════════

sales['rev_cv']  = sales['rev_std']  / sales['rev_mean']
sales['cogs_cv'] = sales['cogs_std'] / sales['cogs_mean']

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=sales['Date'], y=sales['rev_cv'],
    mode='lines',
    name='Revenue CV',
    line=dict(color='#2196F3', width=1.5)
))

fig3.add_trace(go.Scatter(
    x=sales['Date'], y=sales['cogs_cv'],
    mode='lines',
    name='COGS CV',
    line=dict(color='#F44336', width=1.5)
))

# Reference line at CV = 1 (std = mean, very high volatility)
fig3.add_hline(y=1.0, line_dash='dash', line_color='grey',
               annotation_text='CV = 1 (std equals mean)', opacity=0.6)

fig3.update_layout(
    title='<b>Coefficient of Variation — Revenue vs COGS (30-day window)</b><br>'
          '<sup>CV = std/mean; scale-independent volatility comparison. '
          'Higher CV = more unpredictable relative to its own average</sup>',
    xaxis_title='Date',
    yaxis_title='Coefficient of Variation',
    hovermode='x unified',
    width=1100, height=450,
    legend=dict(orientation='h', y=1.08)
)
fig3.show()


# ══════════════════════════════════════════════════════════════
# CHART 4 — Annual Variance Summary (box plots by year)
# Shows which years had the most unpredictable revenue
# ══════════════════════════════════════════════════════════════

sales['year'] = sales['Date'].dt.year

fig4 = make_subplots(rows=1, cols=2,
    subplot_titles=('Revenue Distribution by Year', 'COGS Distribution by Year'))

years = sorted(sales['year'].unique())
colors = ['#2196F3'] * len(years)

for i, year in enumerate(years):
    yr_data = sales[sales['year'] == year]

    fig4.add_trace(go.Box(
        y=yr_data['Revenue'],
        name=str(year),
        marker_color='#2196F3',
        showlegend=False,
        boxpoints='outliers'
    ), row=1, col=1)

    fig4.add_trace(go.Box(
        y=yr_data['COGS'],
        name=str(year),
        marker_color='#F44336',
        showlegend=False,
        boxpoints='outliers'
    ), row=1, col=2)

fig4.update_layout(
    title='<b>Revenue & COGS Distribution by Year</b><br>'
          '<sup>Box height = variance; dots = outlier days; '
          'shrinking boxes post-2019 confirm revenue decline & reduced volatility</sup>',
    width=1100, height=550
)
fig4.update_yaxes(title_text='Revenue (VND)', row=1, col=1)
fig4.update_yaxes(title_text='COGS (VND)', row=1, col=2)
fig4.show()